# Compare Output Segmentations

Let's compare U-Net outputs with NVAE-Corrector outputs.

In [ ]:
import os

# Change this variable if the root folder name has been changed
root_dir = "nvae-shape-encoding"
current_dir = os.getcwd()

if not current_dir.endswith(root_dir):
    %cd ..

assert os.getcwd().endswith(root_dir)

We don't have to perform inference for U-Net, since the predictions are already
saved for the Mask-to-Mask pipeline.

Update `model_path` to choose the model.

In [ ]:
import lightning as L
import torch

from arch.nvae_corrector.nvae_corrector import NVAECorrector
from utils.const import SEED
from data_modules.acdc import ACDC3DWithPredictedMaskDataModule, ACDCDataModule
from utils.utils import setup_device

model_path = "logs/nvae_corrector_acdc/unet-only/checkpoints/epoch=61-step=13268.ckpt"

# Setup device
device = setup_device()
print(f"Device: {device}")

# Seed
L.seed_everything(SEED)

# Load data
data_module = ACDC3DWithPredictedMaskDataModule()

# Load model
model = NVAECorrector.load_from_checkpoint(model_path)

# Reseed after preprocessing data
L.seed_everything(SEED)

In [ ]:
loader_test = data_module.test_dataloader()
y, y_pred, _, _ = next(iter(loader_test))
print(y.shape, y_pred.shape)

## View Segmentations

Let's seek the worst U-Net predictions in terms of anatomical validity, and see
how NVAE-Corrector improves them.

In [ ]:
# Do not use a data loader, since scans are encapsulated. Furthermore, not
# further preprocessing is done in Dataset.__getitem__.

data = data_module.data_test

print(data.scans[0].shape)
print(data.masks[0].shape)
print(data.masks_pred[0].shape)

print(len(data))

In [ ]:
from utils.anatomical_validity_checker import AnatomicalValidityChecker
from utils.eval import compute_dice_score
from utils.utils import discretise

loader_test = data_module.test_dataloader()

worst_x = []
worst_y = []
worst_y_pred = []
worst_y_hat = []

with torch.no_grad():
    model.eval()
    model.to(device)
    
    for i in range(len(data)):
        # 3D data
        xs = data.scans[i]
        ys = data.masks[i]
        y_preds = data.masks_pred[i]
        
        for x, y, y_pred in zip(xs, ys, y_preds):
            # Individual 2D slice
            x = x.unsqueeze(0)
            y = y.unsqueeze(0)
            y_pred = y_pred.unsqueeze(0)

            y = y.to(device)
            y_pred = y_pred.to(device)

            y_hat_logits, _, _, _, _ = model(y_pred)
            y_hat_onehot = discretise(y_hat_logits)
            
            # Condition: DSC < 0.4 for U-Net prediction
            
            # # Compute DSC for U-Net prediction
            # dice_score: torch.Tensor = compute_dice_score(y, y_pred, device=device)
            
            # if dice_score.item() < 0.4:
            #     worst_x.append(x)
            #     worst_y.append(y)
            #     worst_y_pred.append(y_pred)
            #     worst_y_hat.append(y_hat_onehot)
            
            # Condition: Anatomically invalid U-Net prediction
            
            for discretised_y_hat in y_pred:
                AV = AnatomicalValidityChecker(discretised_y_hat)
                # There is only 1 case where there are 3+ violations
                if AV.count_violations() > 1:
                    worst_x.append(x)
                    worst_y.append(y)
                    worst_y_pred.append(y_pred)
                    worst_y_hat.append(y_hat_onehot)

print(len(worst_x))

In [ ]:
# Choose the samples to view

samples_idx = torch.randperm(len(worst_x))[:32]

worst_x_subset = torch.stack([worst_x[i] for i in samples_idx]).squeeze(1)
worst_y_subset = torch.stack([worst_y[i] for i in samples_idx]).squeeze(1)
worst_y_pred_subset = torch.stack([worst_y_pred[i] for i in samples_idx]).squeeze(1)
worst_y_hat_subset = torch.stack([worst_y_hat[i] for i in samples_idx]).squeeze(1)

worst_x_subset.shape, worst_y_subset.shape, worst_y_pred_subset.shape, worst_y_hat_subset.shape

In [ ]:
from utils.eval import get_samples_and_reconstructions_pixel_diff


ground_truth, predictions, reconstruction_pixel_error = get_samples_and_reconstructions_pixel_diff(
    worst_y_subset,
    worst_y_pred_subset,
    return_reconstructions=True,
)

_, corrections, reconstruction_pixel_error_correcitons = get_samples_and_reconstructions_pixel_diff(
    worst_y_subset,
    worst_y_hat_subset,
    return_reconstructions=True,
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from torchvision.utils import make_grid

from utils.colourmap import GIRIDIS

fig, ax = plt.subplots(1, 4, figsize=(24, 144))

# Column 1: Scans

scans = worst_x_subset.cpu().float()
scans = make_grid(scans, nrow=1, padding=2)
scans = np.transpose(scans.numpy(), (1, 2, 0))

ax[0].axis("off")
ax[0].imshow(scans)
ax[0].set_title("Scan", fontsize=36, pad=36)

# Column 2: Ground truth

ground_truth = ground_truth.cpu().float()
ground_truth_grid = make_grid(ground_truth, nrow=1, padding=2, normalize=True)
ground_truth_grid = ground_truth_grid[0]

ax[1].axis("off")
ax[1].imshow(ground_truth_grid, cmap=GIRIDIS)
ax[1].set_title("Ground Truth", fontsize=36, pad=36)

# Column 3: Predictions

predictions = predictions.cpu().float()
predictions_grid = make_grid(predictions, nrow=1, padding=2, normalize=True)
predictions_grid = predictions_grid[0]

ax[2].axis("off")
ax[2].imshow(predictions_grid, cmap=GIRIDIS)
ax[2].set_title("U-Net Segmentation", fontsize=36, pad=36)

# Column 4: Corrections

corrections = corrections.cpu().float()
corrections_grid = make_grid(corrections, nrow=1, padding=2, normalize=True)
corrections_grid = corrections_grid[0]

ax[3].axis("off")
ax[3].imshow(corrections_grid, cmap=GIRIDIS)
ax[3].set_title("NVAE-Corrector", fontsize=36, pad=36)